# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far, models have been trained with default hyperparameters. In this lab, we prepare the features, establish a baseline Random Forest model, and use **GridSearchCV** to select hyperparameters through cross-validation.

The test set is held out until final evaluation. Preprocessing is placed inside a scikit-learn pipeline so imputers, scaling, and one-hot encoding are fitted only on each training fold, reducing data leakage.

In [1]:
# Libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


## Feature preparation and selection

- Create `CabinDeck` and `CabinSide` from the original `Cabin` field.
- Remove `PassengerId` and `Name` because they are identifiers, and replace the raw `Cabin` field with the extracted features.
- Impute missing values, scale numerical features, and one-hot encode categorical features.
- Split the data into training and test sets using stratification to preserve the target class proportions.

The preprocessing pipeline will be fitted only on the training data.
- Convert pandas missing values (`pd.NA`) to `np.nan`: `SimpleImputer` cannot handle `pd.NA` in object columns and raises `TypeError: boolean value of NA is ambiguous`.


In [3]:
# 1. Prepare the data and select useful features

# Work on a copy so the original dataset remains unchanged.
df = spaceship.copy()

# The target is whether a passenger was transported.
target = "Transported"
df = df.dropna(subset=[target])
y = df[target].astype(bool)

# Extract useful information from Cabin before dropping the original column.
# Cabin is usually formatted as Deck/Number/Side, for example "B/0/P".
# Note: we use the object dtype (missing values stay np.nan) instead of the "string" dtype (which creates pd.NA).
cabin_parts = df["Cabin"].astype("object").str.split("/", expand=True)
df["CabinDeck"] = cabin_parts[0] if cabin_parts.shape[1] > 0 else np.nan
df["CabinSide"] = cabin_parts[2] if cabin_parts.shape[1] > 2 else np.nan

# PassengerId and Name are identifiers rather than directly useful predictors.
# Cabin has been replaced by its extracted Deck and Side features.
X = df.drop(columns=["Transported", "PassengerId", "Name", "Cabin"], errors="ignore")

# Safety net: convert any pandas missing value (pd.NA) into np.nan so SimpleImputer works.
def na_to_nan(data):
    data = data.copy()
    for col in data.columns:
        if pd.api.types.is_numeric_dtype(data[col]) and not pd.api.types.is_bool_dtype(data[col]):
            data[col] = data[col].astype("float64")
        else:
            data[col] = data[col].astype("object").where(data[col].notna(), np.nan)
    return data

X = na_to_nan(X)

# Identify numerical and categorical columns for preprocessing.
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

# Impute missing values, scale numerical variables, and one-hot encode categories.
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)
print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)


Training set: (6954, 12)
Test set: (1739, 12)
Numerical features: ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
Categorical features: ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'CabinDeck', 'CabinSide']


## Baseline model

We use a `RandomForestClassifier` as the model to tune. A random forest combines predictions from multiple decision trees and can model non-linear relationships. We record its baseline performance before searching for better hyperparameters.

In [4]:
# 2. Train the best-performing model to tune: Random Forest
# The preprocessing is inside the Pipeline to prevent data leakage.
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(random_state=42, n_jobs=-1))
])

rf_pipeline.fit(X_train, y_train)
y_pred_baseline = rf_pipeline.predict(X_test)

print("Baseline Random Forest accuracy:", round(accuracy_score(y_test, y_pred_baseline), 4))


Baseline Random Forest accuracy: 0.7993


## Baseline evaluation

Accuracy gives the overall proportion of correct predictions. The classification report also provides precision, recall, and F1-score for each class; the confusion matrix shows correct and incorrect predictions by class.

In [5]:
# 3. Evaluate the baseline model
print("Baseline accuracy:", round(accuracy_score(y_test, y_pred_baseline), 4))
print("\nClassification report:")
print(classification_report(y_test, y_pred_baseline, zero_division=0))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred_baseline))


Baseline accuracy: 0.7993

Classification report:
              precision    recall  f1-score   support

       False       0.77      0.84      0.81       863
        True       0.83      0.76      0.79       876

    accuracy                           0.80      1739
   macro avg       0.80      0.80      0.80      1739
weighted avg       0.80      0.80      0.80      1739

Confusion matrix:
[[725 138]
 [211 665]]


## Grid Search

Grid Search tests the specified hyperparameter combinations and uses cross-validation on the training data to estimate performance. The held-out test set is not used to choose the parameters.

For this lab, we use `GridSearchCV` with 5-fold cross-validation and accuracy as the selection metric. The best estimator is refitted on the complete training set.

### Define the hyperparameters

We tune the number of trees (`n_estimators`), tree depth (`max_depth`), minimum samples required to split a node (`min_samples_split`), minimum samples per leaf (`min_samples_leaf`), and the number of features considered at each split (`max_features`).

In [6]:
# 4. Define the hyperparameters to tune
# Keep the grid moderate so the search remains practical on a laptop.
param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2],
    "model__max_features": ["sqrt", 0.8]
}

print("Number of parameter combinations:", (
    len(param_grid["model__n_estimators"])
    * len(param_grid["model__max_depth"])
    * len(param_grid["model__min_samples_split"])
    * len(param_grid["model__min_samples_leaf"])
    * len(param_grid["model__max_features"])
))


Number of parameter combinations: 48


### Run Grid Search

The search evaluates every combination using 5-fold cross-validation. Because the grid contains 48 combinations, this means up to 240 model fits. This may take a few minutes depending on your computer.

In [7]:
# 5. Run Grid Search with cross-validation
grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    scoring="accuracy",
    cv=5,
    n_jobs=-1,
    verbose=1,
    refit=True
)

grid_search.fit(X_train, y_train)

print("Best cross-validation accuracy:", round(grid_search.best_score_, 4))
print("Best hyperparameters:")
print(grid_search.best_params_)


Fitting 5 folds for each of 48 candidates, totalling 240 fits


Best cross-validation accuracy: 0.8004
Best hyperparameters:
{'model__max_depth': 10, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 1, 'model__min_samples_split': 5, 'model__n_estimators': 200}


## Final evaluation and conclusion

Compare the tuned model's test accuracy with the baseline and review precision, recall, F1-score, and the confusion matrix. Report the actual values printed by your notebook; tuning does not guarantee that test accuracy will increase.

In [8]:
# 6. Evaluate the tuned model on the held-out test set
best_model = grid_search.best_estimator_
y_pred_tuned = best_model.predict(X_test)

baseline_accuracy = accuracy_score(y_test, y_pred_baseline)
tuned_accuracy = accuracy_score(y_test, y_pred_tuned)

print("Baseline test accuracy:", round(baseline_accuracy, 4))
print("Tuned test accuracy:", round(tuned_accuracy, 4))
print("Accuracy change (percentage points):", round((tuned_accuracy - baseline_accuracy) * 100, 2))

print("\nTuned model classification report:")
print(classification_report(y_test, y_pred_tuned, zero_division=0))
print("Tuned model confusion matrix:")
print(confusion_matrix(y_test, y_pred_tuned))

if tuned_accuracy > baseline_accuracy:
    print("\nConclusion: hyperparameter tuning improved test accuracy on this split.")
elif tuned_accuracy < baseline_accuracy:
    print("\nConclusion: tuning did not improve test accuracy on this split; compare metrics and consider another model/grid.")
else:
    print("\nConclusion: test accuracy is unchanged after tuning.")


Baseline test accuracy: 0.7993
Tuned test accuracy: 0.8016
Accuracy change (percentage points): 0.23

Tuned model classification report:
              precision    recall  f1-score   support

       False       0.79      0.81      0.80       863
        True       0.81      0.79      0.80       876

    accuracy                           0.80      1739
   macro avg       0.80      0.80      0.80      1739
weighted avg       0.80      0.80      0.80      1739

Tuned model confusion matrix:
[[699 164]
 [181 695]]

Conclusion: hyperparameter tuning improved test accuracy on this split.
